# Data leakage (vazamento de dados)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
df = pd.read_csv('heart_tratado.csv', sep = ';', encoding='utf-8')

In [3]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289.0,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180.0,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283.0,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214.0,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195.0,0,Normal,122,N,0.0,Up,0


In [4]:
X = df.drop('HeartDisease', axis=1) #previsores 
y = df['HeartDisease']#alvo

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0) 

In [6]:
#Identificar colunas categóricas e numéricas
categoricas = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
numericas = [col for col in X.columns if col not in categoricas]

In [7]:
# pipeline com OneHotEncoder
preprocess_onehot = ColumnTransformer([
    ('num',Pipeline([
        #substitui valores missing pela média
        #('imp', SimpleImputer(strategy='mean')), 
        #('scaler', StandardScaler())#padronização
        ('scaler', MinMaxScaler())#normalização
    ]),numericas),

    ('cal', Pipeline([
        # ('imp', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categoricas)
])


# Pipeline com OrdinalEncoder
# OrdinalEncoder é similar ao LabelEncoder, porém tem as vantagens de
# atuar num conjunto de atributos, ao invés de atuação individual e
# é recomendado para uso em atributos previsores ou variáveis independentes.
preprocess_ordinal = ColumnTransformer([
    ('num', Pipeline([
        # ('imp', SimpleImputer(strategy='mean')),
        #('scaler', StandardScaler())  # Padronização
        ('scaler', MinMaxScaler())  # Normalização
    ]), numericas),

    ('cat', Pipeline([
        # ('imp', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder())
    ]), categoricas)
])

In [8]:
pipeline_onehot = Pipeline([
    ('pre', preprocess_onehot),
    ('clf', XGBClassifier(max_depth=2, learning_rate=0.05, n_estimators=250, random_state=3))
])

In [9]:
pipeline_ordinal = Pipeline([
    ('pre', preprocess_ordinal),
    ('clf', XGBClassifier(max_depth=2, learning_rate=0.05, n_estimators=250, random_state=3))
])

In [10]:
# Avaliação em dados de teste
print("\nAvaliação com dados de teste (OneHot):")
pipeline_onehot.fit(X_train, y_train)
y_pred1 = pipeline_onehot.predict(X_test)
print(f"Acurácia: {accuracy_score(y_test, y_pred1):.4f}")
print(confusion_matrix(y_test, y_pred1))
print(classification_report(y_test, y_pred1))


Avaliação com dados de teste (OneHot):
Acurácia: 0.8659
[[102  19]
 [ 18 137]]
              precision    recall  f1-score   support

           0       0.85      0.84      0.85       121
           1       0.88      0.88      0.88       155

    accuracy                           0.87       276
   macro avg       0.86      0.86      0.86       276
weighted avg       0.87      0.87      0.87       276



In [11]:
print("\nAvaliação com dados de teste (Ordinal):")
pipeline_ordinal.fit(X_train, y_train)
y_pred2 = pipeline_ordinal.predict(X_test)
print(f"Acurácia: {accuracy_score(y_test, y_pred2):.4f}")
print(confusion_matrix(y_test, y_pred2))
print(classification_report(y_test, y_pred2))


Avaliação com dados de teste (Ordinal):
Acurácia: 0.8587
[[101  20]
 [ 19 136]]
              precision    recall  f1-score   support

           0       0.84      0.83      0.84       121
           1       0.87      0.88      0.87       155

    accuracy                           0.86       276
   macro avg       0.86      0.86      0.86       276
weighted avg       0.86      0.86      0.86       276



In [12]:
# Comparação via validação cruzada
print("OneHot:")
scores1 = cross_val_score(pipeline_onehot, X, y, cv=5)
print(f"Acurácia média: {scores1.mean():.4f}")

print("\nOrdinal:")
scores2 = cross_val_score(pipeline_ordinal, X, y, cv=5)
print(f"Acurácia média: {scores2.mean():.4f}")

OneHot:
Acurácia média: 0.8331

Ordinal:
Acurácia média: 0.8364
